# M-Detector Debugging and Tuning Notebook
## 0. Setup and Imports

In [1]:
# %%
import os
import sys
import yaml
import numpy as np
import matplotlib.pyplot as plt
import k3d
import h5py
from tqdm.notebook import tqdm # Use tqdm.notebook for better notebook integration
from pyquaternion import Quaternion
import logging
from typing import Dict, List, Optional, Tuple, Any

# --- Add Project Root to sys.path ---
# Adjust this path if your notebook is located elsewhere relative to the project root
NOTEBOOK_DIR = os.path.abspath('') 
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR) 
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print(f"Added project root to sys.path: {PROJECT_ROOT}")

# --- Import Custom Modules (after adding project root) ---
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import Box as NuScenesDataClassesBox

# Core M-Detector components
from src.core.m_detector.base import MDetector
from src.core.depth_image import DepthImage
from src.core.constants import OcclusionResult, POINT_LABEL_DTYPE
from src.core.debug_collector import PointDebugCollector

# Data utilities
from src.data_utils.nuscenes_helper import get_scene_sweep_data_sequence, get_lidar_sweep_data
from src.utils.transformations import transform_points_numpy
from src.utils.validation_utils import get_gt_dynamic_points_for_sweep # For loading GT points

# Visualization (basic K3D helper, can be expanded)
# from src.visualization.k3d_visualizer import visualize_sweep_k3d # If using the full function
# For more granular control, we might define simpler K3D plotting functions here or import primitives

print("Imports complete.")

# --- Configure Logging for M-Detector ---
# Get the root logger for M-Detector if specific modules have their own
# For simplicity, let's configure logging for the 'src' module or specific M-Detector modules.
# Example: Configure logging for the MDetector class itself
mdetector_logger = logging.getLogger('src.core.m_detector.base') # Or 'src.core.m_detector' for the whole module
mdetector_logger.setLevel(logging.DEBUG)
processing_logger = logging.getLogger('src.core.m_detector.processing')
processing_logger.setLevel(logging.DEBUG)
# map_consistency_logger = logging.getLogger('src.core.m_detector.map_consistency')
# map_consistency_logger.setLevel(logging.DEBUG)

print("Logging configured.")

# %%
# --- Load Configuration ---
try:
    config_path = os.path.join(PROJECT_ROOT, 'config/m_detector_config.yaml')
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    print(f"Configuration loaded successfully from: {config_path}")
except FileNotFoundError:
    print(f"ERROR: Config file not found at {config_path}. Please check the path.")
    config = {} 
except Exception as e:
    print(f"Error loading config: {e}")
    config = {}

# --- Initialize NuScenes ---
nusc = None
if 'nuscenes' in config and config.get('nuscenes', {}).get('dataroot'):
    try:
        nusc = NuScenes(
            version=config['nuscenes']['version'],
            dataroot=config['nuscenes']['dataroot'],
            verbose=config.get('nuscenes', {}).get('verbose_init', False)
        )
        print(f"NuScenes SDK initialized (Version: {config['nuscenes']['version']}).")
    except Exception as e:
        print(f"Error initializing NuScenes SDK: {e}")
else:
    print("ERROR: NuScenes 'dataroot' or 'version' missing in config. Cannot initialize NuScenes SDK.")

# --- Define Paths for GT Labels and M-Detector Outputs (HDF5) ---
# These should ideally come from your config or be set explicitly here
GT_LABELS_HDF5_DIR = config.get('nuscenes', {}).get('label_path', os.path.join(PROJECT_ROOT, 'data/gt_labels_hdf5'))
MDET_RESULTS_HDF5_DIR = config.get('mdetector_output', {}).get('save_path', os.path.join(PROJECT_ROOT, 'output/mdet_results_hdf5'))

print(f"GT Labels HDF5 Directory: {GT_LABELS_HDF5_DIR}")
print(f"M-Detector Results HDF5 Directory: {MDET_RESULTS_HDF5_DIR}")

# --- Scene and Sweep Selection ---
TARGET_SCENE_NAME = 'scene-0103' 

SKIP_SWEEPS = 40
MAX_SWEEPS = 20
TARGET_SWEEP_INDEX_IN_SCENE = 10 # SKIP_SWEEPS + TARGET_SWEEP_INDEX_IN_SCENE = actual sweep index

target_scene_rec = None
if nusc:
    for scene in nusc.scene:
        if scene['name'] == TARGET_SCENE_NAME:
            target_scene_rec = scene
            break
    if target_scene_rec:
        print(f"Target Scene: '{target_scene_rec['name']}' (Token: {target_scene_rec['token']})")
        
        scene_sweeps_full_sequence = list(get_scene_sweep_data_sequence(nusc, target_scene_rec['token']))
        scene_sweeps_full_sequence = scene_sweeps_full_sequence[SKIP_SWEEPS:SKIP_SWEEPS+MAX_SWEEPS]
        if 0 <= TARGET_SWEEP_INDEX_IN_SCENE < len(scene_sweeps_full_sequence):
            TARGET_SWEEP_DATA_DICT = scene_sweeps_full_sequence[TARGET_SWEEP_INDEX_IN_SCENE]
            TARGET_LIDAR_SD_TOKEN = TARGET_SWEEP_DATA_DICT['lidar_sd_token']
            print(f"Target Sweep Index: {TARGET_SWEEP_INDEX_IN_SCENE}, Token: {TARGET_LIDAR_SD_TOKEN[:8]}..., Timestamp: {TARGET_SWEEP_DATA_DICT['timestamp']}")
        else:
            print(f"ERROR: TARGET_SWEEP_INDEX_IN_SCENE ({TARGET_SWEEP_INDEX_IN_SCENE}) is out of bounds for scene '{TARGET_SCENE_NAME}' (0-{len(scene_sweeps_full_sequence)-1}).")
            TARGET_LIDAR_SD_TOKEN = None
            TARGET_SWEEP_DATA_DICT = None
    else:
        print(f"ERROR: Scene '{TARGET_SCENE_NAME}' not found.")
        TARGET_LIDAR_SD_TOKEN = None
        TARGET_SWEEP_DATA_DICT = None
else:
    print("NuScenes SDK not initialized. Cannot select scene/sweep.")
    TARGET_LIDAR_SD_TOKEN = None
    TARGET_SWEEP_DATA_DICT = None



Added project root to sys.path: /home/drugge/Unsupervised-Moving-Point-Detection/m_detector_python
Imports complete.
Logging configured.
Configuration loaded successfully from: /home/drugge/Unsupervised-Moving-Point-Detection/m_detector_python/config/m_detector_config.yaml
NuScenes SDK initialized (Version: v1.0-mini).
GT Labels HDF5 Directory: /home/drugge/staff-umbrella/TeamHolgerResearch/drugge/nuscenes_labels_interpolated
M-Detector Results HDF5 Directory: /home/drugge/staff-umbrella/TeamHolgerResearch/drugge/m_detector_output
Target Scene: 'scene-0103' (Token: fcbccedd61424f1b85dcbf8f897f9754)
Target Sweep Index: 10, Token: 9181fe90..., Timestamp: 1533151606048630


In [2]:

def load_gt_labels_for_sweep_from_hdf5(
    nusc_obj: NuScenes,
    scene_name: str, # Scene name to construct filename
    sweep_data_dict_for_gt: Dict, # Original sweep data for the target sweep
    gt_hdf5_base_dir: str,
    velocity_threshold_gt: float
) -> Optional[Dict[str, np.ndarray]]:
    """Loads GT dynamic/static points for a specific sweep from the scene's HDF5 GT file."""
    gt_scene_hdf5_filename = f"gt_point_labels_{scene_name}.h5"
    gt_labels_scene_hdf5_filepath = os.path.join(gt_hdf5_base_dir, gt_scene_hdf5_filename)

    if not os.path.exists(gt_labels_scene_hdf5_filepath):
        print(f"GT HDF5 file not found: {gt_labels_scene_hdf5_filepath}")
        return None
    
    # We need all_points_global for get_gt_dynamic_points_for_sweep if it's to return 'unlabeled'
    # For just dynamic/static, it might not be strictly needed by the function if it re-extracts from HDF5
    # Let's assume we pass None if we only care about dynamic/static from the HDF5 structure itself.
    # However, get_gt_dynamic_points_for_sweep uses all_points_global for the 'unlabeled' key.
    # For visualization, we'll typically plot the raw sweep points and overlay GT/MDet.
    # The function get_gt_dynamic_points_for_sweep is designed to match GT labels to a given point cloud.
    # Let's ensure we have the points that these GT labels correspond to.
    # The GT HDF5 file ('all_gt_point_labels') stores global coordinates of the *original* full point cloud.
    
    # For this helper, let's simplify: it returns the points *as stored in the GT HDF5 file*
    # and then we can match them later if needed, or use their stored global coords directly.
    
    points_from_gt_file = {}
    with h5py.File(gt_labels_scene_hdf5_filepath, 'r') as hf:
        all_tokens_bytes = hf['sweep_lidar_sd_tokens'][:]
        target_token_bytes = sweep_data_dict_for_gt['lidar_sd_token'].encode('utf-8')
        
        matches = np.where(all_tokens_bytes == target_token_bytes)[0]
        if not matches.size > 0:
            print(f"Sweep token {sweep_data_dict_for_gt['lidar_sd_token']} not found in GT HDF5 {gt_labels_scene_hdf5_filepath}")
            return None
        idx_in_token_list = matches[0]

        indices_array = hf['gt_point_labels_indices'][:]
        start_idx = indices_array[idx_in_token_list]
        end_idx = indices_array[idx_in_token_list + 1]
        
        point_labels_for_sweep_structured = hf['all_gt_point_labels'][start_idx:end_idx]

    if point_labels_for_sweep_structured.shape[0] > 0:
        gt_points_global = np.stack((point_labels_for_sweep_structured['x'],
                                     point_labels_for_sweep_structured['y'],
                                     point_labels_for_sweep_structured['z']), axis=-1)
        
        gt_speed_sq = point_labels_for_sweep_structured['velocity_x']**2 + point_labels_for_sweep_structured['velocity_y']**2
        is_valid_instance_mask = (point_labels_for_sweep_structured['instance_token'] != b'')
        
        dynamic_mask = (gt_speed_sq >= velocity_threshold_gt**2) & is_valid_instance_mask
        static_mask = (~dynamic_mask) & is_valid_instance_mask
        
        points_from_gt_file['gt_dynamic_pts'] = gt_points_global[dynamic_mask]
        points_from_gt_file['gt_static_pts'] = gt_points_global[static_mask]
        points_from_gt_file['gt_all_labeled_pts_structured'] = point_labels_for_sweep_structured # For more detailed inspection
    else:
        points_from_gt_file['gt_dynamic_pts'] = np.empty((0,3))
        points_from_gt_file['gt_static_pts'] = np.empty((0,3))
        points_from_gt_file['gt_all_labeled_pts_structured'] = np.empty(0, dtype=POINT_LABEL_DTYPE)
        
    return points_from_gt_file

def load_mdet_results_for_sweep_from_hdf5(
    mdet_scene_hdf5_filepath: str,
    target_sweep_token: str
) -> Optional[Dict[str, np.ndarray]]:
    """Loads M-Detector classified points for a specific sweep from its HDF5 results file."""
    if not os.path.exists(mdet_scene_hdf5_filepath):
        print(f"M-Detector HDF5 file not found: {mdet_scene_hdf5_filepath}")
        return None

    with h5py.File(mdet_scene_hdf5_filepath, 'r') as hf:
        all_tokens_bytes = hf['sweep_lidar_sd_tokens'][:]
        target_token_bytes = target_sweep_token.encode('utf-8')
        
        matches = np.where(all_tokens_bytes == target_token_bytes)[0]
        if not matches.size > 0:
            print(f"Sweep token {target_sweep_token} not found in MDet HDF5 {mdet_scene_hdf5_filepath}")
            return None
        idx_in_token_list = matches[0]

        indices_array = hf['points_predictions_indices'][:]
        start_idx = indices_array[idx_in_token_list]
        end_idx = indices_array[idx_in_token_list + 1]
        
        preds_structured = hf['all_points_predictions'][start_idx:end_idx]

    if preds_structured.shape[0] > 0:
        points_global = np.stack((preds_structured['x'], preds_structured['y'], preds_structured['z']), axis=-1)
        labels = preds_structured['mdet_label']
        
        mdet_results = {
            'mdet_dynamic_pts': points_global[labels == OcclusionResult.OCCLUDING_IMAGE.value],
            'mdet_occluded_pts': points_global[labels == OcclusionResult.OCCLUDED_BY_IMAGE.value],
            'mdet_empty_pts': points_global[labels == OcclusionResult.EMPTY_IN_IMAGE.value],
            'mdet_undetermined_pts': points_global[labels == OcclusionResult.UNDETERMINED.value],
            'mdet_all_pts_global': points_global, # All points processed by MDet for this sweep
            'mdet_all_labels': labels # Corresponding labels
        }
    else:
        mdet_results = {
            'mdet_dynamic_pts': np.empty((0,3)), 'mdet_occluded_pts': np.empty((0,3)),
            'mdet_empty_pts': np.empty((0,3)), 'mdet_undetermined_pts': np.empty((0,3)),
            'mdet_all_pts_global': np.empty((0,3)), 'mdet_all_labels': np.empty(0, dtype=int)
        }
    return mdet_results

# K3D Visualization Helper
def create_k3d_plot_with_points(points_dict: Dict[str, Tuple[np.ndarray, int, float]], 
                                plot_title: str = "LiDAR Sweep",
                                background_color=0xAAAAAA,
                                grid_visible=True, camera_auto_fit=True):
    """
    Creates a K3D plot with multiple sets of points.
    points_dict: {'label': (points_array_Nx3, color_hex, point_size)}
    """
    plot = k3d.plot(name=plot_title, grid_visible=grid_visible, camera_auto_fit=camera_auto_fit, background_color=background_color)
    for label, (points, color, size) in points_dict.items():
        if points.shape[0] > 0:
            plot += k3d.points(positions=points.astype(np.float32), color=color, point_size=size, shader='3d', name=label)
    return plot

print("Helper functions defined.")


Helper functions defined.


In [ ]:

# --- Instantiate M-Detector ---
if config and nusc: # Ensure config and nusc are loaded
    # --- Configuration Override for Debugging (Optional) ---
    # You can temporarily override config values here for specific tests
    # For example, to test map consistency:
    # config['map_consistency_check']['enabled'] = True
    # config['map_consistency_check']['static_labels_for_map_check'] = ["OCCLUDED_BY_IMAGE"] # More conservative
    # config['map_consistency_check']['map_consistency_threshold_count'] = 1 
    # config['map_consistency_check']['map_consistency_threshold_ratio'] = 0.0 # Effectively OR
    # config['occlusion_check']['epsilon_depth_occlusion'] = 0.3 # Example: try smaller

    detector = MDetector(config)
    print("M-Detector instantiated.")

    # --- Process Sweeps up to and including the Target Sweep ---
    # This section is crucial for setting up the M-Detector's state (DI library)
    # correctly for debugging the target sweep.
    
    processed_target_di: Optional[DepthImage] = None
    idx_of_processed_target_di: Optional[int] = None

    if TARGET_LIDAR_SD_TOKEN and target_scene_rec:
        print(f"\nProcessing sweeps for scene '{target_scene_rec['name']}' up to target sweep...")
        
        # Reset detector state for this scene
        detector.reset_scene_state()
        
        # Iterate through all sweeps in the scene sequence
        for i, sweep_data in enumerate(tqdm(scene_sweeps_full_sequence, desc="Feeding Sweeps")):
            # Add sweep to M-Detector
            added_di = detector.add_sweep_and_create_depth_image(
                points_lidar_frame=sweep_data['points_sensor_frame'],
                T_global_lidar=sweep_data['T_global_lidar'],
                lidar_timestamp=sweep_data['timestamp'],
                lidar_sd_token=sweep_data['lidar_sd_token']
            )
            
            # If this is the target sweep, we want to process IT specifically AFTER this loop
            # and inspect its state.
            # For sweeps *before* the target, we call decide_and_process_frame to let M-Detector
            # build its library and process frames as it normally would.
            if sweep_data['lidar_sd_token'] != TARGET_LIDAR_SD_TOKEN:
                # Process causally if it's not the target sweep yet
                # This ensures historical DIs are processed and their labels are set,
                # which is important if map consistency relies on those labels.
                if not detector.use_bidirectional: # Assuming causal focus for now
                    # Find the index of the newly added DI (it's the last one)
                    if detector.depth_image_library:
                        idx_to_process_causal = len(detector.depth_image_library) - 1
                        # Only process if it's newer than the last processed one
                        if detector.timestamp_of_last_processed_di is None or \
                           added_di.timestamp > detector.timestamp_of_last_processed_di:
                            if idx_to_process_causal >= detector.min_sweeps_for_processing -1 : # Basic readiness
                                detector.timestamp_of_last_processed_di = added_di.timestamp
                                detector._process_causal_di_wrapper(idx_to_process_causal)
            else: # This IS the target sweep
                # We've added it. We will process it manually in the next cell.
                processed_target_di = added_di
                try:
                    idx_of_processed_target_di = detector.depth_image_library._images.index(processed_target_di)
                except ValueError:
                    print(f"ERROR: Target DI (Token: {TARGET_LIDAR_SD_TOKEN}) was added but not found in library. This is unexpected.")
                    idx_of_processed_target_di = -1 # Mark as error
                print(f"\nTarget sweep {TARGET_LIDAR_SD_TOKEN[:8]} added to M-Detector. Ready for manual processing and inspection.")
                break # Stop feeding sweeps after adding the target
        
        if not processed_target_di:
            print(f"ERROR: Target sweep {TARGET_LIDAR_SD_TOKEN} was not reached or added correctly.")

    else:
        print("Target sweep or scene not defined. Skipping M-Detector processing setup.")
        detector = None
else:
    print("M-Detector or NuScenes not initialized. Cannot proceed.")
    detector = None


# 1. Define the points you want to investigate for a particular run
points_to_investigate = [100, 3028] # Add other points as needed

# 2. Create and attach the debug collector BEFORE processing the relevant DI/sweep
debug_collector = PointDebugCollector(points_to_trace=points_to_investigate)
detector.set_debug_collector(debug_collector)

# --- Manually Trigger Processing for the Target DI ---
if detector and processed_target_di and idx_of_processed_target_di is not None and idx_of_processed_target_di != -1:
    print(f"Manually processing target DI: {processed_target_di.timestamp} (Index in lib: {idx_of_processed_target_di})")
    
    # Ensure the MDetector's internal timestamp is updated to reflect this processing
    # This is important if any internal logic in MDetector relies on timestamp_of_last_processed_di
    # However, process_and_label_di itself doesn't use it, but decide_and_process_frame does.
    # For direct call, it's less critical, but good practice if we were mixing.
    # detector.timestamp_of_last_processed_di = processed_target_di.timestamp # Update MDetector's state

    # Call the causal processing function directly
    # This will populate:
    #   processed_target_di.raw_occlusion_results_vs_history
    #   processed_target_di.mdet_labels_for_points
    result_dict = detector.process_and_label_di(processed_target_di, idx_of_processed_target_di)
    
    if result_dict.get('success'):
        print(f"Processing successful for target DI. Points labeled: {result_dict.get('points_labeled')}")
        print(f"Label counts: {result_dict.get('label_counts')}")
        
        # Now processed_target_di is fully populated for causal logic.
        # We can inspect its attributes.
    else:
        print(f"Processing FAILED for target DI. Reason: {result_dict.get('reason')}")
else:
    print("Cannot process target DI: M-Detector not ready, or target DI not identified/found in library.")

print("\n--- Debug Collector Data ---")
if debug_collector: # Check if it was set
    for pt_idx in points_to_investigate:
        trace_data = debug_collector.get_trace_data(pt_idx)
        if trace_data:
            print(f"\nTrace for Point Index {pt_idx}:")
            for key, value in trace_data.items():
                print(f"  {key}: {value}")
        else:
            print(f"\nNo trace data collected for Point Index {pt_idx} (was it processed?).")

    # 5. Important: Clear or detach the collector if you don't want it for subsequent runs
    detector.set_debug_collector(None) 
    # or debug_collector.clear_data() / debug_collector.set_points_to_trace([])
else:
    print("No debug collector was attached.")


pt_idx_to_debug = 100

if processed_target_di and processed_target_di.original_points_global_coords is not None and \
   pt_idx_to_debug < len(processed_target_di.original_points_global_coords):
    
    point_100_global = processed_target_di.original_points_global_coords[pt_idx_to_debug]
    print(f"Debugging Point {pt_idx_to_debug} Global Coords: {point_100_global}")

    if idx_of_processed_target_di is not None and idx_of_processed_target_di > 0:
        immediate_past_di_idx = idx_of_processed_target_di - 1
        immediate_past_di = detector.depth_image_library.get_image_by_index(immediate_past_di_idx)

        if immediate_past_di:
            print(f"Target Historical DI for check: Timestamp {immediate_past_di.timestamp}")
            
            # --- Start of copied logic from check_occlusion_pixel_level ---
            # Parameters from MDetector instance (self) that are used:
            # self.neighbor_search_pixels_v
            # self.neighbor_search_pixels_h
            # self.epsilon_depth_occlusion
            
            print("\n1. Projecting current_point_global into historical_depth_image's frame...")
            # Note: historical_depth_image.project_point_to_pixel_indices is a method of DepthImage
            point_in_hist_di_frame, sph_coords_curr, pixel_indices_in_hist_di = \
                immediate_past_di.project_point_to_pixel_indices(point_100_global)

            print(f"  Point in Hist DI Frame: {point_in_hist_di_frame}")
            print(f"  Spherical Coords (phi, theta, d_curr): {sph_coords_curr}")
            print(f"  Projected Pixel Indices (v, h): {pixel_indices_in_hist_di}")

            if pixel_indices_in_hist_di is None or sph_coords_curr is None:
                print("  RESULT: UNDETERMINED (Point projects outside historical DI's FoV or invalid projection)")
                # End of debug for this case
            else:
                v_idx_curr_proj, h_idx_curr_proj = pixel_indices_in_hist_di
                d_curr = sph_coords_curr[2] 
                print(f"  Depth of Current Point (d_curr): {d_curr}")

                print("\n2. Gathering min/max depths from projected pixel and neighbors...")
                v_start = max(0, v_idx_curr_proj - detector.neighbor_search_pixels_v)
                v_end = min(immediate_past_di.num_pixels_v, v_idx_curr_proj + detector.neighbor_search_pixels_v + 1)
                h_start = max(0, h_idx_curr_proj - detector.neighbor_search_pixels_h)
                h_end = min(immediate_past_di.num_pixels_h, h_idx_curr_proj + detector.neighbor_search_pixels_h + 1)
                
                print(f"  Region bounds (v_start, v_end, h_start, h_end): ({v_start}, {v_end}, {h_start}, {h_end})")
                
                region_min_depths_slice = immediate_past_di.pixel_min_depth[v_start:v_end, h_start:h_end]
                region_max_depths_slice = immediate_past_di.pixel_max_depth[v_start:v_end, h_start:h_end]
                region_counts_slice = immediate_past_di.pixel_count[v_start:v_end, h_start:h_end]
                
                # For detailed view of the region:
                # print(f"  Region Min Depths in Hist DI:\n{region_min_depths_slice}")
                # print(f"  Region Max Depths in Hist DI:\n{region_max_depths_slice}")
                # print(f"  Region Point Counts in Hist DI:\n{region_counts_slice}")
                
                has_data_mask = region_counts_slice > 0
                found_data_in_region = np.any(has_data_mask)
                print(f"  Found data in region of historical DI: {found_data_in_region}")
                
                if not found_data_in_region:
                    print(f"  RESULT: EMPTY_IN_IMAGE (No points in the historical DI's pixel region)")
                    # End of debug for this case
                else:
                    min_depth_in_region = np.min(region_min_depths_slice[has_data_mask])
                    max_depth_in_region = np.max(region_max_depths_slice[has_data_mask])
                    print(f"  Min Depth in Hist Region: {min_depth_in_region}")
                    print(f"  Max Depth in Hist Region: {max_depth_in_region}")
                    print(f"  Epsilon Depth Occlusion: {detector.epsilon_depth_occlusion}")

                    print("\n3. Checking occlusion conditions...")
                    # Condition for OCCLUDED_BY_IMAGE: d_curr > max_depth_in_region + epsilon
                    cond_occluded_thresh = max_depth_in_region + detector.epsilon_depth_occlusion
                    print(f"  Is d_curr ({d_curr:.3f}) > max_depth_in_region + eps ({max_depth_in_region:.3f} + {detector.epsilon_depth_occlusion:.3f} = {cond_occluded_thresh:.3f})? {d_curr > cond_occluded_thresh}")
                    if d_curr > cond_occluded_thresh:
                        print(f"  RESULT: OCCLUDED_BY_IMAGE")
                        # End of debug for this case
                    
                    # Condition for OCCLUDING_IMAGE: d_curr < min_depth_in_region - epsilon
                    cond_occluding_thresh = min_depth_in_region - detector.epsilon_depth_occlusion
                    print(f"  Is d_curr ({d_curr:.3f}) < min_depth_in_region - eps ({min_depth_in_region:.3f} - {detector.epsilon_depth_occlusion:.3f} = {cond_occluding_thresh:.3f})? {d_curr < cond_occluding_thresh}")
                    if not (d_curr > cond_occluded_thresh) and (d_curr < cond_occluding_thresh): # Check only if not already decided
                         print(f"  RESULT: OCCLUDING_IMAGE")
                         # End of debug for this case
                    
                    # If neither of the above:
                    if not (d_curr > cond_occluded_thresh) and not (d_curr < cond_occluding_thresh):
                        print(f"  RESULT: UNDETERMINED (d_curr is within or too close to the historical depth range)")
                        # End of debug for this case
        else:
            print(f"ERROR: Could not retrieve immediate_past_di (Timestamp {immediate_past_di.timestamp if immediate_past_di else 'N/A'}).")
    else:
        print(f"ERROR: Target DI is the first DI (idx={idx_of_processed_target_di}), no immediate past DI available for this check.")
else:
    print(f"ERROR: Could not get Point {pt_idx_to_debug} coordinates.")


# --- Notebook Cell for Getting GT for Point 100 and Preparing Visualization ---

# Ensure 'nusc', 'TARGET_SCENE_NAME', 'TARGET_SWEEP_DATA_DICT', 
# 'processed_target_di', 'detector', 'idx_of_processed_target_di',
# 'GT_LABELS_HDF5_DIR', and 'config' (for velocity_threshold_gt) are available.

pt_idx_to_debug = 100
point_100_global = None
gt_status_for_point_100 = "GT Unknown" # Default

if processed_target_di and processed_target_di.original_points_global_coords is not None and \
   pt_idx_to_debug < len(processed_target_di.original_points_global_coords):
    
    point_100_global = processed_target_di.original_points_global_coords[pt_idx_to_debug]
    print(f"Point {pt_idx_to_debug} Global Coords: {point_100_global}")

    # --- Get GT classification for all points in the target sweep ---
    gt_scene_hdf5_filename = f"gt_point_labels_{TARGET_SCENE_NAME}.h5"
    gt_labels_scene_hdf5_filepath = os.path.join(GT_LABELS_HDF5_DIR, gt_scene_hdf5_filename)
    
    velocity_threshold_gt = config.get('evaluation', {}).get('velocity_threshold_gt', 0.1) # Get from config

    if os.path.exists(gt_labels_scene_hdf5_filepath):
        print(f"Loading GT labels from: {gt_labels_scene_hdf5_filepath}")
        gt_info_for_sweep = get_gt_dynamic_points_for_sweep(
            nusc=nusc,
            sweep_data_dict=TARGET_SWEEP_DATA_DICT, # Original sweep data for the target sweep
            all_points_global=processed_target_di.original_points_global_coords,
            gt_labels_scene_hdf5_path=gt_labels_scene_hdf5_filepath,
            velocity_threshold=velocity_threshold_gt
        )
        
        if gt_info_for_sweep:
            if pt_idx_to_debug in gt_info_for_sweep.get('gt_dynamic_indices', []):
                gt_status_for_point_100 = "GT Dynamic"
            elif pt_idx_to_debug in gt_info_for_sweep.get('gt_static_indices', []):
                gt_status_for_point_100 = "GT Static"
            elif pt_idx_to_debug in gt_info_for_sweep.get('unlabeled_indices', []):
                gt_status_for_point_100 = "GT Unlabeled"
            else:
                gt_status_for_point_100 = "GT Status Undetermined (not in any GT category)"
            print(f"GT Status for Point {pt_idx_to_debug}: {gt_status_for_point_100}")
            
            # For broader context visualization (optional)
            gt_dynamic_points_target_sweep = processed_target_di.original_points_global_coords[gt_info_for_sweep.get('gt_dynamic_indices', [])]
            gt_static_points_target_sweep = processed_target_di.original_points_global_coords[gt_info_for_sweep.get('gt_static_indices', [])]
            gt_unlabeled_points_target_sweep = processed_target_di.original_points_global_coords[gt_info_for_sweep.get('unlabeled_indices', [])]

        else:
            print(f"Could not retrieve GT info for sweep {TARGET_SWEEP_DATA_DICT['lidar_sd_token']}")
    else:
        print(f"GT HDF5 file not found: {gt_labels_scene_hdf5_filepath}. Cannot determine GT for Point {pt_idx_to_debug}.")
else:
    print(f"ERROR: Could not get Point {pt_idx_to_debug} coordinates from processed_target_di.")

# --- Prepare Historical Points for Visualization ---
historical_context_points_global = np.empty((0,3), dtype=np.float32)

if idx_of_processed_target_di is not None and idx_of_processed_target_di > 0:
    immediate_past_di_idx = idx_of_processed_target_di - 1
    immediate_past_di = detector.depth_image_library.get_image_by_index(immediate_past_di_idx)

    # CORRECTED LINE HERE:
    if immediate_past_di and immediate_past_di.pixel_original_indices is not None: # Changed attribute name
        # Get the projection details from the previous debug cell's output (or re-calculate for robustness)
        # For this example, let's use the known values from your previous output for Point 100:
        # Projected Pixel Indices (v, h): (7, 896)
        # Region bounds (v_start, v_end, h_start, h_end): (6, 9, 894, 899)
        v_proj_hist, h_proj_hist = (7, 896) # Point 100's projection in immediate_past_di
        
        # Use the same neighbor search window as in occlusion check
        v_start_hist = max(0, v_proj_hist - detector.neighbor_search_pixels_v)
        v_end_hist = min(immediate_past_di.num_pixels_v, v_proj_hist + detector.neighbor_search_pixels_v + 1)
        h_start_hist = max(0, h_proj_hist - detector.neighbor_search_pixels_h)
        h_end_hist = min(immediate_past_di.num_pixels_h, h_proj_hist + detector.neighbor_search_pixels_h + 1)
        
        print(f"\nHistorical DI (TS: {immediate_past_di.timestamp}): Extracting points from region v=[{v_start_hist}-{v_end_hist-1}], h=[{h_start_hist}-{h_end_hist-1}]")

        collected_hist_points_indices = []
        for v_idx in range(v_start_hist, v_end_hist):
            for h_idx in range(h_start_hist, h_end_hist):
                pixel_key = (v_idx, h_idx) # Use pixel_key for defaultdict
                # CORRECTED LOGIC HERE:
                if pixel_key in immediate_past_di.pixel_original_indices:
                    collected_hist_points_indices.extend(immediate_past_di.pixel_original_indices[pixel_key])
        
        if collected_hist_points_indices:
            # Remove duplicates if any point index appeared in multiple pixels (unlikely with typical LiDAR projection)
            unique_hist_indices = np.unique(collected_hist_points_indices)
            if immediate_past_di.original_points_global_coords is not None:
                historical_context_points_global = immediate_past_di.original_points_global_coords[unique_hist_indices]
                print(f"  Found {len(historical_context_points_global)} unique historical points in the region.")
            else:
                print("  Historical DI has no global coordinates stored.")
        else:
            print("  No historical points found in the specified pixel region.")
    else:
        # Add a check for pixel_original_indices specifically if immediate_past_di exists
        if immediate_past_di and immediate_past_di.pixel_original_indices is None:
             print("Could not get immediate past DI pixel_original_indices (it's None).")
        elif not immediate_past_di:
             print("Could not get immediate past DI.")

else:
    print("Target DI is the first DI, or index not available. Cannot get historical context points.")


# --- Visualization ---
if point_100_global is not None:
    plot_points_dict = {}

    # Point 100
    point_100_color = 0xff0000 # Red (MDet Undetermined)
    # If you want to reflect GT status in color even if "Undetermined"
    # if gt_status_for_point_100 == "GT Unlabeled" or "Undetermined" in gt_status_for_point_100:
    #     point_100_color = 0xFFD700 # Gold for GT Undetermined/Unlabeled

    plot_points_dict[f'Point {pt_idx_to_debug} (MDet: UNDET, GT: {gt_status_for_point_100})'] = (
        point_100_global.reshape(1,3), point_100_color, 0.4 # Make it significantly larger
    )

    # Historical Context Points
    if historical_context_points_global.shape[0] > 0:
        plot_points_dict['Historical Context Pts (Immediate Past)'] = (
            historical_context_points_global, 0x0000FF, 0.3 # BLUE, slightly larger
        )

    # --- ADD ALL POINTS FROM THE CURRENT TARGET SWEEP FOR CONTEXT ---
    if processed_target_di and processed_target_di.original_points_global_coords is not None:
        all_current_sweep_points = processed_target_di.original_points_global_coords
        
        # Optional: Downsample if too many points for performance
        # num_context_points = 10000
        # if all_current_sweep_points.shape[0] > num_context_points:
        #     choice_indices = np.random.choice(all_current_sweep_points.shape[0], num_context_points, replace=False)
        #     context_plot_points = all_current_sweep_points[choice_indices]
        # else:
        #     context_plot_points = all_current_sweep_points
        context_plot_points = all_current_sweep_points # Plot all for now

        plot_points_dict['All Points (Current Target Sweep)'] = (
            context_plot_points, 0xAAAAAA, 0.25 # Light Grey, very small
        )
    # --- END ADDING CURRENT SWEEP POINTS ---
        
    plot = create_k3d_plot_with_points(
        plot_points_dict,
        plot_title=f"P{pt_idx_to_debug} (MDet:UNDET, GT:{gt_status_for_point_100}) vs Hist", # Shorter title
        camera_auto_fit=True # TRY True first with all current sweep points
    )
    
    # If camera_auto_fit=True doesn't give a good view, then manually set it focusing on Point 100
    # but ensure the view is wide enough to see the context points.
    if not plot.camera_auto_fit: # Or if you prefer manual control
        view_offset = 20 # Increase this distance
        plot.camera = [
            point_100_global[0] + view_offset, point_100_global[1] + view_offset, point_100_global[2] + view_offset / 2,
            point_100_global[0], point_100_global[1], point_100_global[2],
            0, 0, 1
        ]
        plot.camera_fov = 60 # Wider FOV

    plot.display()
else:
    print("Cannot visualize: Point 100 data not available.")
    
    
# --- Notebook Cell for Investigating raw_occlusion_results_vs_history for Point 100 ---

# Ensure 'processed_target_di', 'detector', 'pt_idx_to_debug', 
# and 'OcclusionResult' are available in your notebook scope.

if processed_target_di is None:
    print("ERROR: processed_target_di is None. Please ensure it's loaded and processed.")
# ... (other initial checks for raw_occlusion_results_vs_history and pt_idx_to_debug bounds) ...
elif idx_of_processed_target_di is None: # Make sure this index is available from your setup
    print("ERROR: idx_of_processed_target_di is None. This is needed to replicate MDetector's DI selection.")
else:
    print(f"Investigating raw_occlusion_results_vs_history for Point {pt_idx_to_debug} in DI (TS: {processed_target_di.timestamp}):")

    # --- Parameters used by MDetector.process_and_label_di for raw_occlusion_results_vs_history ---
    event_test_cfg = detector.config.get('event_tests', {})
    # This N determines the number of columns in raw_occlusion_results_vs_history
    # and how many historical DIs were iterated over.
    n_hist_for_test3_logic = event_test_cfg.get('test1_N_depth_images', 5) 
    print(f"  MDetector's process_and_label_di used N_historical_DIs: {n_hist_for_test3_logic} (from event_tests.test1_N_depth_images)")

    actual_historical_dis_for_test3 = []
    # Replicate the loop from MDetector.process_and_label_di
    for k_hist_idx in range(n_hist_for_test3_logic):
        # Get the (k_hist_idx)-th older DI (0 is immediate predecessor)
        # idx_of_processed_target_di is the index of current_di in the library when it was processed.
        historical_di_lib_index = idx_of_processed_target_di - 1 - k_hist_idx
        
        if historical_di_lib_index < 0:
            # print(f"    Debug: Ran out of historical DIs in library at k_hist_idx={k_hist_idx} (lib_idx={historical_di_lib_index}).")
            break 

        historical_di_object = detector.depth_image_library.get_image_by_index(historical_di_lib_index)
        
        if historical_di_object:
            actual_historical_dis_for_test3.append(historical_di_object)
        else:
            # This case should ideally not happen if the library didn't change and indices are correct.
            # print(f"    Debug: Could not retrieve DI at library index {historical_di_lib_index} for k_hist_idx={k_hist_idx}.")
            # To maintain the same number of "slots" as raw_occlusion_results_vs_history might have,
            # you could append a None or a placeholder if a DI is missing, but this indicates a deeper issue.
            # For now, we only append if found. This might lead to a mismatch if some DIs became None.
            pass

    # The order in actual_historical_dis_for_test3 is now:
    # [immediate_past, one_before_immediate_past, ..., up to Nth_past]
    # This directly corresponds to columns 0, 1, ..., N-1 of raw_occlusion_results_vs_history.

    print(f"  Retrieved {len(actual_historical_dis_for_test3)} historical DIs for comparison using MDetector's indexing logic.")

    point_100_raw_results = processed_target_di.raw_occlusion_results_vs_history[pt_idx_to_debug, :]

    if point_100_raw_results.shape[0] == 0:
        print("  Point 100 has no raw occlusion results stored (array is empty).")
    # Check if the number of retrieved DIs matches the number of results for this point.
    # The number of columns in raw_occlusion_results_vs_history is fixed by n_hist_for_test3_logic.
    # len(actual_historical_dis_for_test3) might be less if the library didn't have enough past DIs.
    elif point_100_raw_results.shape[0] != n_hist_for_test3_logic:
        print(f"  WARNING: Number of results for point ({point_100_raw_results.shape[0]}) "
              f"does not match config N_historical_DIs ({n_hist_for_test3_logic}). This is unexpected.")
    elif len(actual_historical_dis_for_test3) == 0 and n_hist_for_test3_logic > 0:
        print(f"  WARNING: Retrieved 0 historical DIs, but expected to check against {n_hist_for_test3_logic} based on config.")
        # Print raw results anyway
        for i_res in range(point_100_raw_results.shape[0]):
            result_val = point_100_raw_results[i_res]
            try:
                result_enum = OcclusionResult(result_val)
                print(f"  vs Hist DI (Original Column {i_res}): Raw Value={result_val}, Result={result_enum.name} (Historical DI object not retrieved for this slot)")
            except ValueError:
                print(f"  vs Hist DI (Original Column {i_res}): Raw Value={result_val}, Result=Unknown (Historical DI object not retrieved for this slot)")

    else:
        print(f"\n  Results for Point {pt_idx_to_debug} against up to {n_hist_for_test3_logic} historical DIs (most recent past first):")
        occluding_count_for_pt100 = 0
        
        # Iterate up to the number of results we have for the point, or number of DIs found, whichever is smaller for printing.
        # The raw_occlusion_results_vs_history array has n_hist_for_test3_logic columns.
        # actual_historical_dis_for_test3 contains the DIs that were actually available.
        
        for i_col in range(point_100_raw_results.shape[0]): # Iterate through all columns in the results array
            result_val = point_100_raw_results[i_col]
            result_enum = OcclusionResult(result_val) # Assume valid value
            
            if result_enum == OcclusionResult.OCCLUDING_IMAGE:
                occluding_count_for_pt100 +=1
            
            if i_col < len(actual_historical_dis_for_test3):
                historical_di = actual_historical_dis_for_test3[i_col] # This DI corresponds to this column
                print(f"  - vs Hist DI (TS: {historical_di.timestamp:.0f}, LibIdx: {idx_of_processed_target_di - 1 - i_col}, "
                      f"Age: {(processed_target_di.timestamp - historical_di.timestamp)/1e6:.3f}s ago): "
                      f"Col {i_col}, Raw Val={result_val}, Result={result_enum.name}")
            else:
                # This column in raw_occlusion_results_vs_history corresponds to a historical DI that was not available
                # (e.g., current_di was too early in the sequence). The result should be UNDETERMINED.
                print(f"  - vs Hist DI (Not available in library for this slot): "
                      f"Col {i_col}, Raw Val={result_val}, Result={result_enum.name}")
        
        print(f"\n  Calculated occluding_count for Point {pt_idx_to_debug} from these results: {occluding_count_for_pt100}")
        
        min_consistency_dynamic_raw = detector.config.get('event_tests', {}).get('test1_M1_threshold', 2)
        passed_event_test_for_pt100 = occluding_count_for_pt100 >= min_consistency_dynamic_raw
        print(f"  min_consistency_dynamic_raw (M1_threshold) from config: {min_consistency_dynamic_raw}")
        print(f"  Did Point {pt_idx_to_debug} pass event test based on these results? {passed_event_test_for_pt100}")

        if passed_event_test_for_pt100:
            print("\n  To further debug why it passed the event test (was OCCLUDING_IMAGE):")
            print("\n  To further debug why it passed the event test (was OCCLUDING_IMAGE):")
            print("  1. Identify one of the historical DIs above where the Result was OCCLUDING_IMAGE.")
            print("  2. Get the global coordinates of Point 100 from 'processed_target_di'.")
            print("  3. Manually call 'detector._check_occlusion_pixel_level()' or replicate its logic: ")
            print("     - Project Point 100 into that specific historical DI.")
            print("     - Get the min/max depths in the projected pixel region of that historical DI.")
            print("     - Compare Point 100's depth (d_curr) against those min/max depths + epsilon.")
            print("     - See why d_curr < min_depth_in_hist_region - epsilon_depth_occlusion was True.")

# --- START: Added section for detailed check ---
if passed_event_test_for_pt100: # Check if the point actually passed the test
    print(f"\n--- Detailed Occlusion Check for Point {pt_idx_to_debug} where it was OCCLUDING_IMAGE ---")

    # Find the first instance where the result was OCCLUDING_IMAGE
    # The 'actual_historical_dis_for_test3' list is aligned with the columns.
    col_to_debug = -1
    for i_col_check in range(point_100_raw_results.shape[0]):
        if OcclusionResult(point_100_raw_results[i_col_check]) == OcclusionResult.OCCLUDING_IMAGE:
            if i_col_check < len(actual_historical_dis_for_test3):
                col_to_debug = i_col_check
                break
            else:
                print(f"  OCCLUDING_IMAGE found at Col {i_col_check}, but no corresponding DI object was retrieved (library likely too short at that time). Cannot debug this specific instance.")
    
    if col_to_debug != -1:
        historical_di_to_debug = actual_historical_dis_for_test3[col_to_debug]

        T_global_current = processed_target_di.image_pose_global
        T_global_historical = historical_di_to_debug.image_pose_global
        #  EXTRA CHECKS
        print(f"\n  Ego Pose Comparison:")
        print(f"    Current DI (TS {processed_target_di.timestamp:.0f}) T_global_lidar[2,3] (Z): {T_global_current[2,3]:.4f}")
        print(f"    Hist. DI (TS {historical_di_to_debug.timestamp:.0f}) T_global_lidar[2,3] (Z): {T_global_historical[2,3]:.4f}")

        from scipy.spatial.transform import Rotation as R
        try:
            rot_current = R.from_matrix(T_global_current[:3,:3])
            rot_historical = R.from_matrix(T_global_historical[:3,:3])
            euler_current = rot_current.as_euler('xyz', degrees=True) # roll, pitch, yaw
            euler_historical = rot_historical.as_euler('xyz', degrees=True)
            print(f"    Current DI Euler (Roll, Pitch, Yaw): {euler_current[0]:.2f}, {euler_current[1]:.2f}, {euler_current[2]:.2f} deg")
            print(f"    Hist. DI Euler (Roll, Pitch, Yaw): {euler_historical[0]:.2f}, {euler_historical[1]:.2f}, {euler_historical[2]:.2f} deg")
            print(f"    Pitch Change (Current - Hist): {euler_current[1] - euler_historical[1]:.2f} deg (Negative means nose down)")
        except Exception as e:
            print(f"    Error calculating Euler angles: {e}")

        print(f"\n  Inspecting points in the historical DI's region {v_start}-{v_end-1}, {h_start}-{h_end-1}:")
        hist_region_point_indices = []
        for v_reg_idx in range(v_start, v_end):
            for h_reg_idx in range(h_start, h_end):
                pixel_key = (v_reg_idx, h_reg_idx)
                indices_in_pixel = historical_di_to_debug.pixel_original_indices.get(pixel_key, [])
                hist_region_point_indices.extend(indices_in_pixel)

        if not hist_region_point_indices:
            print("    No original point indices found in historical region (problem with pixel_original_indices).")
        elif historical_di_to_debug.original_points_global_coords is None or \
            historical_di_to_debug.local_sph_coords_for_points is None:
            print("    Historical DI is missing original_points_global_coords or local_sph_coords_for_points.")
        else:
            unique_hist_region_indices = np.unique(hist_region_point_indices)
            print(f"    Found {len(unique_hist_region_indices)} unique points in historical region.")
            
            hist_region_points_global = historical_di_to_debug.original_points_global_coords[unique_hist_region_indices]
            hist_region_points_depths_in_hist_frame = historical_di_to_debug.local_sph_coords_for_points[unique_hist_region_indices, 2]
            
            for i_pt, original_idx in enumerate(unique_hist_region_indices[:5]): # Print first 5
                print(f"      Hist Pt Idx {original_idx}: Global Z: {hist_region_points_global[i_pt, 2]:.3f}, Depth in Hist DI: {hist_region_points_depths_in_hist_frame[i_pt]:.3f}")
            
            if len(hist_region_points_depths_in_hist_frame) > 0:
                idx_min_depth_hist_region = np.argmin(hist_region_points_depths_in_hist_frame)
                actual_min_depth_pt_global_z = hist_region_points_global[idx_min_depth_hist_region, 2]
                print(f"    Point contributing to min_depth_in_hist_region ({hist_region_points_depths_in_hist_frame[idx_min_depth_hist_region]:.3f}) has Global Z: {actual_min_depth_pt_global_z:.3f}")
                # Compare its global Z with Point 100's global Z. They should be very similar if both are flat ground.
        #  END EXTRA CHECKS
        print(f"Focusing on historical DI at Column {col_to_debug} (TS: {historical_di_to_debug.timestamp}, "
              f"Original LibIdx: {idx_of_processed_target_di - 1 - col_to_debug}) " 
              f"where Point {pt_idx_to_debug} was OCCLUDING_IMAGE.")

        if processed_target_di.original_points_global_coords is None or \
           pt_idx_to_debug >= len(processed_target_di.original_points_global_coords):
            print(f"  ERROR: Cannot get global coordinates for Point {pt_idx_to_debug} from processed_target_di.")
        else:
            point_to_check_global = processed_target_di.original_points_global_coords[pt_idx_to_debug]
            print(f"  Global Coords of Point {pt_idx_to_debug}: {point_to_check_global}")

            # Replicate detector._check_occlusion_pixel_level() logic:
            # Or call: detector.check_occlusion_pixel_level(point_to_check_global, historical_di_to_debug)
            # For clarity, let's replicate the core logic here:

            print("\n  1. Projecting current_point_global into the chosen historical_depth_image's frame...")
            point_in_hist_di_frame, sph_coords_curr, pixel_indices_in_hist_di = \
                historical_di_to_debug.project_point_to_pixel_indices(point_to_check_global)

            print(f"    Point in Hist DI Frame: {point_in_hist_di_frame}")
            print(f"    Spherical Coords (phi, theta, d_curr): {sph_coords_curr}")
            print(f"    Projected Pixel Indices (v, h): {pixel_indices_in_hist_di}")

            if pixel_indices_in_hist_di is None or sph_coords_curr is None:
                print("    RESULT: UNDETERMINED (Point projects outside historical DI's FoV or invalid projection)")
            else:
                v_idx_curr_proj, h_idx_curr_proj = pixel_indices_in_hist_di
                d_curr = sph_coords_curr[2] 
                print(f"    Depth of Current Point (d_curr): {d_curr}")

                print("\n  2. Gathering min/max depths from projected pixel and neighbors in the chosen historical DI...")
                v_start = max(0, v_idx_curr_proj - detector.neighbor_search_pixels_v)
                v_end = min(historical_di_to_debug.num_pixels_v, v_idx_curr_proj + detector.neighbor_search_pixels_v + 1)
                h_start = max(0, h_idx_curr_proj - detector.neighbor_search_pixels_h)
                h_end = min(historical_di_to_debug.num_pixels_h, h_idx_curr_proj + detector.neighbor_search_pixels_h + 1)
                
                print(f"    Region bounds (v_start, v_end, h_start, h_end): ({v_start}, {v_end}, {h_start}, {h_end})")
                
                # Ensure pixel data arrays are not None
                if historical_di_to_debug.pixel_min_depth is None or \
                   historical_di_to_debug.pixel_max_depth is None or \
                   historical_di_to_debug.pixel_count is None:
                    print("    ERROR: Historical DI pixel data (min_depth, max_depth, or count) is None.")
                else:
                    region_min_depths_slice = historical_di_to_debug.pixel_min_depth[v_start:v_end, h_start:h_end]
                    region_max_depths_slice = historical_di_to_debug.pixel_max_depth[v_start:v_end, h_start:h_end]
                    region_counts_slice = historical_di_to_debug.pixel_count[v_start:v_end, h_start:h_end]
                    
                    has_data_mask = region_counts_slice > 0
                    found_data_in_region = np.any(has_data_mask)
                    print(f"    Found data in region of historical DI: {found_data_in_region}")
                    
                    if not found_data_in_region:
                        print(f"    RESULT: EMPTY_IN_IMAGE (No points in the historical DI's pixel region)")
                    else:
                        min_depth_in_region = np.min(region_min_depths_slice[has_data_mask])
                        max_depth_in_region = np.max(region_max_depths_slice[has_data_mask])
                        print(f"    Min Depth in Hist Region: {min_depth_in_region}")
                        print(f"    Max Depth in Hist Region: {max_depth_in_region}")
                        print(f"    Epsilon Depth Occlusion: {detector.epsilon_depth_occlusion}")

                        print("\n  3. Checking occlusion conditions...")
                        cond_occluded_thresh = max_depth_in_region + detector.epsilon_depth_occlusion
                        is_occluded = d_curr > cond_occluded_thresh
                        print(f"    Is d_curr ({d_curr:.3f}) > max_depth_in_region + eps ({max_depth_in_region:.3f} + {detector.epsilon_depth_occlusion:.3f} = {cond_occluded_thresh:.3f})? {is_occluded}")
                        
                        cond_occluding_thresh = min_depth_in_region - detector.epsilon_depth_occlusion
                        is_occluding = d_curr < cond_occluding_thresh
                        print(f"    Is d_curr ({d_curr:.3f}) < min_depth_in_region - eps ({min_depth_in_region:.3f} - {detector.epsilon_depth_occlusion:.3f} = {cond_occluding_thresh:.3f})? {is_occluding}")
                        
                        final_check_result = OcclusionResult.UNDETERMINED
                        if is_occluded:
                            final_check_result = OcclusionResult.OCCLUDED_BY_IMAGE
                        elif is_occluding: # Only if not already decided as occluded
                            final_check_result = OcclusionResult.OCCLUDING_IMAGE
                        
                        print(f"    MANUAL CHECK RESULT for this DI: {final_check_result.name}")
                        original_result_from_history = OcclusionResult(point_100_raw_results[col_to_debug])
                        if final_check_result == original_result_from_history:
                            print(f"    This matches the stored result ({original_result_from_history.name}).")
                        else:
                            print(f"    WARNING: This MISMATCHES the stored result ({original_result_from_history.name}). Review logic or parameters.")
    elif len(actual_historical_dis_for_test3) > 0 : # Only print if we had DIs but point didn't pass
        print(f"\n  Point {pt_idx_to_debug} did not pass the event test (occluding_count={occluding_count_for_pt100}, threshold={min_consistency_dynamic_raw}). No detailed OCCLUDING_IMAGE check to perform.")

# --- END: Added section ---


# Cell 5.1: Inspect Raw Occlusion Results vs. History (Corrected K3D part)

import numpy as np
from src.core.constants import OcclusionResult # Ensure this is imported
# from src.visualization.k3d_utils import create_k3d_plot_with_points # If you uncomment plotting

print(f"\n--- Inspecting Raw Occlusion Results (vs. History) for Target DI (Timestamp: {processed_target_di.timestamp if processed_target_di else 'N/A'}) ---")

if processed_target_di and processed_target_di.raw_occlusion_results_vs_history is not None:
    history_array = processed_target_di.raw_occlusion_results_vs_history
    num_pts_in_target, num_hist_frames_checked = history_array.shape
    print(f"Shape of raw_occlusion_results_vs_history: {history_array.shape}")

    # ... (rest of the point inspection loop remains the same) ...
    sample_pt_indices_to_inspect = [0, 100, 1000, 5000, 10000] 
    sample_pt_indices_to_inspect = [idx for idx in sample_pt_indices_to_inspect if idx < num_pts_in_target]

    if not sample_pt_indices_to_inspect:
        print("No valid sample point indices to inspect.")
    
    for pt_idx in sample_pt_indices_to_inspect:
        print(f"\nPoint Index {pt_idx}:")
        if processed_target_di.original_points_global_coords is not None and pt_idx < len(processed_target_di.original_points_global_coords):
            print(f"  Global Coords: {processed_target_di.original_points_global_coords[pt_idx]}")
        else:
            print(f"  Global Coords: Not available or index out of bounds.")
            
        history_for_pt = history_array[pt_idx, :]
        try:
            history_readable = [OcclusionResult(val).name for val in history_for_pt]
            print(f"  Raw Occlusion History (vs DI_curr-1, DI_curr-2, ...): {history_readable}")
            
            occluding_count = np.sum(history_for_pt == OcclusionResult.OCCLUDING_IMAGE.value)
            print(f"  Count of OCCLUDING_IMAGE in history: {occluding_count}")
        except ValueError as e:
            print(f"  Error processing history for point {pt_idx}: {e}. Raw values: {history_for_pt}")


    # --- K3D Visualization of Raw Occlusion (Example: vs immediate predecessor) ---
    if detector and detector.depth_image_library and \
       processed_target_di and processed_target_di.timestamp is not None and \
       num_hist_frames_checked > 0: # Ensure history was actually calculated

        # Find the immediate predecessor DI that was used for history_array[:, 0]
        # This would be the newest image in the library with timestamp < processed_target_di.timestamp
        immediate_hist_di_for_plot = None
        latest_past_timestamp = -1.0

        # Iterate through the library to find the DI that would have been the immediate predecessor
        # when raw_occlusion_results_vs_history was calculated.
        # This assumes the library's state hasn't changed drastically *after* processing target_di
        # and *before* running this inspection cell. For robust inspection, it's best if this cell
        # runs immediately after the MDetector processes the target_di.
        
        # Get all images from the library (deque ensures order: oldest to newest)
        # We want the newest one that is older than the current target_di
        # The `get_relevant_past_images` with a very small window or just taking the first one
        # from a sorted list of past images could also work.
        
        # Let's use get_image_by_timestamp with 'before' mode and a very small tolerance
        # to ensure we get the one *just* before.
        # However, the raw occlusion history was built against a specific set of N DIs.
        # The simplest way to get the DI corresponding to history_array[:, 0] is to
        # re-fetch it using the logic that _calculate_raw_occlusion_vs_history would have used.
        
        # The MDetector's _calculate_raw_occlusion_vs_history iterates through
        # self.depth_image_library.get_all_images() in reverse.
        # The first one it encounters that is older than current_di.timestamp is the immediate predecessor.
        
        all_lib_images = detector.depth_image_library.get_all_images()
        potential_predecessors = [
            img for img in all_lib_images if img.timestamp < processed_target_di.timestamp
        ]
        if potential_predecessors:
            # The newest of these past images is the immediate predecessor
            immediate_hist_di_for_plot = max(potential_predecessors, key=lambda img: img.timestamp)
        
        if immediate_hist_di_for_plot and immediate_hist_di_for_plot.original_points_global_coords is not None:
            print(f"Identified immediate historical DI for K3D plot (TS: {immediate_hist_di_for_plot.timestamp})")
            points_for_k3d = {}
            
            raw_occ_vs_imm_hist = history_array[:, 0] 
            
            colors = {
                OcclusionResult.OCCLUDING_IMAGE.value: 0x00FF00, 
                OcclusionResult.OCCLUDED_BY_IMAGE.value: 0xFF0000, 
                OcclusionResult.EMPTY_IN_IMAGE.value: 0xFFFF00,    
                OcclusionResult.UNDETERMINED.value: 0x808080      
            }
            
            for occ_res_enum in OcclusionResult:
                mask = (raw_occ_vs_imm_hist == occ_res_enum.value)
                if np.any(mask):
                    points_for_k3d[f"TargetDI_RawOcc_{occ_res_enum.name}"] = (
                        processed_target_di.original_points_global_coords[mask],
                        colors.get(occ_res_enum.value, 0x000000), 
                        0.05 
                    )
            
            points_for_k3d["HistoricalDI_Points"] = (
                immediate_hist_di_for_plot.original_points_global_coords,
                0x0000FF, 
                0.03 
            )
            
            plot = create_k3d_plot_with_points(points_for_k3d, f"Raw Occlusion: Target DI (TS {processed_target_di.timestamp}) vs Hist DI (TS {immediate_hist_di_for_plot.timestamp})")
            if plot: plot.display()
            print(f"K3D plot for raw occlusion vs immediate history can be generated (code commented out).")
        else:
            print("Immediate historical DI or its points not available for K3D raw occlusion plot.")
    else:
        print("Conditions not met for K3D raw occlusion plot (e.g., no history, library issue, or target DI issue).")
else:
    print("Target DI not processed or no raw occlusion history available.")
    
# Cell 5.2: Inspect Map Consistency Check (MCC)


# Configure logging for the map_consistency module
map_consistency_logger = logging.getLogger('src.core.m_detector.map_consistency')
map_consistency_logger.setLevel(logging.DEBUG) # Set to DEBUG to see all logs

# Add a handler if not already configured (e.g., if running in a fresh kernel)
if not map_consistency_logger.hasHandlers():
    console_handler = logging.StreamHandler(sys.stdout) # Output to notebook console
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    console_handler.setFormatter(formatter)
    map_consistency_logger.addHandler(console_handler)
import numpy as np
import logging # Ensure logging is imported
import sys # Ensure sys is imported
from src.core.constants import OcclusionResult # Ensure this is imported
# Ensure load_gt_labels_for_sweep_from_hdf5 is imported
# from src.data_utils.label_generation import load_gt_labels_for_sweep_from_hdf5

# --- Configure logging for the map_consistency module (if not already done) ---
map_consistency_logger = logging.getLogger('src.core.m_detector.map_consistency')
if not map_consistency_logger.handlers: # Check if handlers are already present
    map_consistency_logger.setLevel(logging.DEBUG) # Set to DEBUG to see all logs
    console_handler = logging.StreamHandler(sys.stdout) # Output to notebook console
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    console_handler.setFormatter(formatter)
    map_consistency_logger.addHandler(console_handler)
    # Prevent duplicate logging if cell is re-run
    map_consistency_logger.propagate = False # Optional: if you see duplicates from root logger

print("\n--- Inspecting Map Consistency Check (MCC) ---")

# --- Load Ground Truth Labels for the Target Sweep for Context ---
gt_labels_target_sweep = None
if nusc and target_scene_rec and TARGET_SWEEP_DATA_DICT and processed_target_di and config:
    if 'GT_LABELS_HDF5_DIR' in locals() or 'GT_LABELS_HDF5_DIR' in globals():
        try:
            # Ensure GT_LABELS_HDF5_DIR is defined (e.g., from a common setup cell)
            # GT_LABELS_HDF5_DIR = "/path/to/your/gt_labels_hdf5_directory" # Example
            gt_labels_target_sweep = load_gt_labels_for_sweep_from_hdf5(
                nusc,
                target_scene_rec['name'],
                TARGET_SWEEP_DATA_DICT,
                GT_LABELS_HDF5_DIR,
                config.get('validation',{}).get('vel_threshold', 0.5)
            )
            if gt_labels_target_sweep:
                print(f"Successfully loaded GT labels for target sweep {TARGET_SWEEP_DATA_DICT['lidar_sd_token'][:8]}.")
            else:
                print(f"Warning: Could not load GT labels for target sweep {TARGET_SWEEP_DATA_DICT['lidar_sd_token'][:8]}.")
        except Exception as e:
            print(f"Error loading GT labels for target sweep: {e}.")
            gt_labels_target_sweep = None
    else:
        print("Warning: GT_LABELS_HDF5_DIR not defined. Cannot load GT labels.")
else:
    print("Conditions not met for loading GT labels (nusc, scene, sweep, DI, or config missing).")


if detector and processed_target_di and \
   processed_target_di.raw_occlusion_results_vs_history is not None and \
   processed_target_di.mdet_labels_for_points is not None and \
   config:
    
    mcc_config = config.get('map_consistency_check', {})
    mcc_enabled = mcc_config.get('enabled', False)
    
    if not mcc_enabled:
        print("Map Consistency Check is DISABLED in config.")
    else:
        if processed_target_di.raw_occlusion_results_vs_history.shape[1] > 0:
            raw_occ_vs_imm_hist_values = processed_target_di.raw_occlusion_results_vs_history[:, 0]
            candidate_for_mcc_mask = (raw_occ_vs_imm_hist_values == OcclusionResult.OCCLUDING_IMAGE.value)
            candidate_indices = np.where(candidate_for_mcc_mask)[0]
            
            print(f"Found {len(candidate_indices)} points initially labeled OCCLUDING_IMAGE (vs immediate history) - candidates for MCC.")

            # --- Define points to inspect, ensuring Point 3028 is included if it's a candidate ---
            # Points from your previous output that were MCC candidates
            sample_mcc_candidate_indices_from_log = [1786, 2142, 3028, 3053, 3054] 
            
            # Filter these to ensure they are actual candidates in the current run
            inspect_indices_mcc = [idx for idx in sample_mcc_candidate_indices_from_log if idx in candidate_indices]
            
            # If Point 3028 was a candidate but not in the list, add it.
            # (It was in your previous log, so it should be picked up by the list above if still a candidate)
            # For robustness, explicitly check and add if it's a candidate:
            pt_3028_idx = 3028
            if pt_3028_idx in candidate_indices and pt_3028_idx not in inspect_indices_mcc:
                inspect_indices_mcc.append(pt_3028_idx)
            
            # Add a few more from the start of candidates if the list is short
            if len(inspect_indices_mcc) < 5:
                additional_candidates = [idx for idx in candidate_indices[:5] if idx not in inspect_indices_mcc]
                inspect_indices_mcc.extend(additional_candidates)
            
            # Remove duplicates and sort for clean output
            inspect_indices_mcc = sorted(list(set(inspect_indices_mcc)))

            if not inspect_indices_mcc:
                if len(candidate_indices) > 0:
                    print("No specific MCC candidates (like 3028) selected for detailed print, but candidates exist.")
                    print(f"First few candidates are: {candidate_indices[:min(5, len(candidate_indices))]}")
                else: # len(candidate_indices) == 0
                    print("No points were candidates for MCC in this sweep.")

            for pt_idx_mcc in inspect_indices_mcc:
                if pt_idx_mcc >= len(processed_target_di.original_points_global_coords):
                    print(f"Skipping pt_idx_mcc {pt_idx_mcc} as it's out of bounds for original_points_global_coords.")
                    continue

                point_global_P_mcc = processed_target_di.original_points_global_coords[pt_idx_mcc]
                print(f"\nInspecting MCC for Point Index {pt_idx_mcc} (Global: {np.round(point_global_P_mcc,3).tolist()}):")
                
                # --- Ground Truth Check ---
                if gt_labels_target_sweep:
                    point_to_check_global_for_gt = point_global_P_mcc
                    dist_thresh_sq = 0.1**2 
                    
                    is_gt_dynamic = False
                    if gt_labels_target_sweep.get('gt_dynamic_pts') is not None and gt_labels_target_sweep['gt_dynamic_pts'].shape[0] > 0:
                        distances_sq_dyn = np.sum((gt_labels_target_sweep['gt_dynamic_pts'] - point_to_check_global_for_gt)**2, axis=1)
                        if np.any(distances_sq_dyn < dist_thresh_sq):
                            is_gt_dynamic = True
                            print(f"  GT INFO: Point {pt_idx_mcc} is CLOSE to a GT DYNAMIC point.")

                    is_gt_static = False
                    if not is_gt_dynamic and gt_labels_target_sweep.get('gt_static_pts') is not None and gt_labels_target_sweep['gt_static_pts'].shape[0] > 0:
                        distances_sq_stat = np.sum((gt_labels_target_sweep['gt_static_pts'] - point_to_check_global_for_gt)**2, axis=1)
                        if np.any(distances_sq_stat < dist_thresh_sq):
                            is_gt_static = True
                            print(f"  GT INFO: Point {pt_idx_mcc} is CLOSE to a GT STATIC point.")
                    
                    if not is_gt_dynamic and not is_gt_static:
                        print(f"  GT INFO: Point {pt_idx_mcc} is UNLABELED as dynamic/static or not close to any GT point.")
                else:
                    print("  GT INFO: Not available for this point/sweep.")
                
                # --- MCC Call and Result ---
                is_consistent, mcc_debug_info = detector.is_map_consistent(
                    point_global_P_mcc, 
                    processed_target_di.timestamp, 
                    check_direction='past',
                    return_debug_info=True 
                )
                print(f"  MCC Result: {'MAP CONSISTENT (Dynamic rejected)' if is_consistent else 'MAP INCONSISTENT (Dynamic confirmed)'}")
                
                # --- Final M-Detector Label ---
                if processed_target_di.mdet_labels_for_points is not None and pt_idx_mcc < len(processed_target_di.mdet_labels_for_points):
                    final_label_val_mcc = processed_target_di.mdet_labels_for_points[pt_idx_mcc]
                    final_label_enum_mcc = OcclusionResult(final_label_val_mcc)
                    print(f"  Final M-Detector Label: {final_label_enum_mcc.name} ({final_label_val_mcc})")
                else:
                    print(f"  Final M-Detector Label: Not available or index out of bounds.")

                # --- Optional: MCC Debug Info (already in your code, kept for completeness) ---
                if mcc_debug_info: # Only print if debug info was returned
                    print(f"  MCC Debug Info:")
                    print(f"    Relevant DIs Checked: {mcc_debug_info.get('relevant_dis_count')}")
                    # print(f"    Relevant DI Timestamps: {mcc_debug_info.get('relevant_dis_timestamps')}") # Can be long
                    print(f"    Static Labels Used for Check: {mcc_debug_info.get('static_labels_for_map_check')}")
                    print(f"    Epsilons (rad, m): Phi={mcc_debug_info.get('epsilon_phi_map_rad'):.3f}, Theta={mcc_debug_info.get('epsilon_theta_map_rad'):.3f}, DepthFwd={mcc_debug_info.get('epsilon_depth_forward_map'):.3f}, DepthBack={mcc_debug_info.get('epsilon_depth_backward_map'):.3f}")
                    print(f"    Consistency Thresholds: Mode={detector.mc_threshold_mode if hasattr(detector, 'mc_threshold_mode') else 'N/A'}, Count={mcc_debug_info.get('consistency_threshold_count')}, Ratio={mcc_debug_info.get('consistency_threshold_ratio')}")
                    print(f"    Final Consistent DIs Count: {mcc_debug_info.get('consistent_count_final')}")
                    print(f"    Total DIs Attempted (Proj&PixelOK): {mcc_debug_info.get('total_checks_attempted_on_dis_final')}")
                    print(f"    Reason for Result: {mcc_debug_info.get('reason_for_result')}")
                    # The MCC_TRACE logs from the logger will provide the ultra-detail if needed.
        else:
            print("No historical data available in raw_occlusion_results_vs_history to check for MCC candidates.")
else:
    print("MCC inspection skipped: Conditions not met (M-Detector, Target DI, mdet_labels, or Config missing).")
    
    
# In your notebook cell for this deep dive

# --- Configuration for Deep Dive ---
target_pt_idx_deep_dive = 3028
historical_di_ts_deep_dive = 1533151605998850 # From MCC log for Pt 3028
# Candidate static point indices from HistDI_A that were compared against Pt 3028
hist_cand_idx1_deep_dive = 3192 
hist_cand_idx2_deep_dive = 3218

# --- Get DI Objects and Key Points ---
current_di_deep_dive = processed_target_di 
# P_current_global is the global coordinate of the target point we are deep-diving on
P_current_global = current_di_deep_dive.original_points_global_coords[target_pt_idx_deep_dive]

historical_di_object_deep_dive = detector.depth_image_library.get_image_by_timestamp(
    historical_di_ts_deep_dive, mode='closest'
)

if historical_di_object_deep_dive is None:
    print(f"ERROR: Could not find historical DI with TS {historical_di_ts_deep_dive}")
    # Handle error or raise an exception if you want to stop
    # For now, let's assume it's found for the rest of the code.
    # If it's None, the subsequent lines will fail. You might want to add an `else:` block.

# --- Prepare Points for K3D Plot ---

# 1. Historical DI's full point cloud (Global Coordinates)
PC_historical_global = historical_di_object_deep_dive.original_points_global_coords

# 2. Global coordinates of the specific candidate static points from the historical DI
P_cand1_hist_global = historical_di_object_deep_dive.original_points_global_coords[hist_cand_idx1_deep_dive]
P_cand2_hist_global = historical_di_object_deep_dive.original_points_global_coords[hist_cand_idx2_deep_dive]

# 3. Project P_current_global into the Historical DI's frame to get its spherical coords
#    as seen by the historical sensor. This is what MCC does internally.
_point_in_hist_di_frame, sph_coords_target_in_hist_frame, _pixel_indices_target_in_hist = \
    historical_di_object_deep_dive.project_point_to_pixel_indices(P_current_global)

if sph_coords_target_in_hist_frame is None:
    print(f"ERROR: Could not project P_current_global into historical DI for sph_coords.")
    # Handle error
    # For now, assume it's successful.

# 4. Create a synthetic global point representing where P_current_global *would have been*
#    if it were part of the historical DI's scan, based on its projected spherical coords.
hist_pose_matrix = historical_di_object_deep_dive.image_pose_global # CORRECTED ATTRIBUTE
# hist_pose_inv_matrix = np.linalg.inv(hist_pose_matrix) # Not strictly needed for this visualization path

phi_proj = sph_coords_target_in_hist_frame[0]
theta_proj = sph_coords_target_in_hist_frame[1]
depth_proj = sph_coords_target_in_hist_frame[2] # This is the ~6.189m for Pt 3028

# Convert these spherical (relative to hist sensor) to Cartesian (relative to hist sensor)
x_proj_local = depth_proj * np.cos(theta_proj) * np.cos(phi_proj)
y_proj_local = depth_proj * np.cos(theta_proj) * np.sin(phi_proj)
z_proj_local = depth_proj * np.sin(theta_proj)
P_proj_local_cartesian_in_hist_sensor = np.array([x_proj_local, y_proj_local, z_proj_local])

# Transform this local projected point to global using historical DI's pose
P_proj_local_homogeneous = np.append(P_proj_local_cartesian_in_hist_sensor, 1)
P_projected_global_homogeneous = hist_pose_matrix @ P_proj_local_homogeneous
P_projected_global = P_projected_global_homogeneous[:3]


# --- K3D Plot Definition ---
points_to_plot_deep_dive = {
    f"CurrentPoint_Global_idx{target_pt_idx_deep_dive}": (P_current_global.reshape(1,3), 0xFF00FF, 0.2), # Magenta
    "HistoricalDI_AllPoints_Global": (PC_historical_global, 0x0000FF, 0.03), # Blue
    f"HistCand1_idx{hist_cand_idx1_deep_dive}_Global": (P_cand1_hist_global.reshape(1,3), 0xFFFF00, 0.15), # Yellow
    f"HistCand2_idx{hist_cand_idx2_deep_dive}_Global": (P_cand2_hist_global.reshape(1,3), 0x00FFFF, 0.15), # Cyan
    "CurrentPoint_ProjectedToHistSensor_Global": (P_projected_global.reshape(1,3), 0xFF0000, 0.2) # Red
}

# --- Add Ground Truth to the Plot ---
gt_static_points_global = None
gt_dynamic_points_global = None

# DEBUGGING GT AVAILABILITY:
print(f"Is 'gt_labels_target_sweep' in locals()? {'gt_labels_target_sweep' in locals()}")
if 'gt_labels_target_sweep' in locals():
    print(f"Type of gt_labels_target_sweep: {type(gt_labels_target_sweep)}")
    if gt_labels_target_sweep: # If it's not None
        print(f"Keys in gt_labels_target_sweep: {list(gt_labels_target_sweep.keys()) if isinstance(gt_labels_target_sweep, dict) else 'Not a dict'}")
        has_gt_static = 'gt_static_pts' in gt_labels_target_sweep and gt_labels_target_sweep['gt_static_pts'] is not None and gt_labels_target_sweep['gt_static_pts'].shape[0] > 0
        has_gt_dynamic = 'gt_dynamic_pts' in gt_labels_target_sweep and gt_labels_target_sweep['gt_dynamic_pts'] is not None and gt_labels_target_sweep['gt_dynamic_pts'].shape[0] > 0
        print(f"  Has GT static points to plot? {has_gt_static}")
        if has_gt_static: print(f"    Shape of gt_static_pts: {gt_labels_target_sweep['gt_static_pts'].shape}")
        print(f"  Has GT dynamic points to plot? {has_gt_dynamic}")
        if has_gt_dynamic: print(f"    Shape of gt_dynamic_pts: {gt_labels_target_sweep['gt_dynamic_pts'].shape}")
    else:
        print("gt_labels_target_sweep is None or empty.")
else:
    print("gt_labels_target_sweep not found in local scope.")


if 'gt_labels_target_sweep' in locals() and gt_labels_target_sweep: 
    if gt_labels_target_sweep.get('gt_static_pts') is not None:
        gt_static_points_global = gt_labels_target_sweep['gt_static_pts']
    if gt_labels_target_sweep.get('gt_dynamic_pts') is not None:
        gt_dynamic_points_global = gt_labels_target_sweep['gt_dynamic_pts']

if 'gt_labels_target_sweep' in locals() and gt_labels_target_sweep:
    all_gt_struct = gt_labels_target_sweep.get('gt_all_labeled_pts_structured')
    if all_gt_struct is not None and all_gt_struct.shape[0] > 0:
        print(f"\nInspecting gt_all_labeled_pts_structured (first 5 rows):")
        print(all_gt_struct[['x', 'y', 'z', 'velocity_x', 'velocity_y', 'instance_token']][:5])

        # You'll need a way to identify points belonging to the moving car.
        # For now, let's look at points near P_current_global (Point 3028)
        # This is a rough way to find them; ideally, you'd use instance tokens if known.
        distances_to_P_current_global = np.linalg.norm(
            np.stack((all_gt_struct['x'], all_gt_struct['y'], all_gt_struct['z']), axis=-1) - P_current_global, axis=1
        )
        close_gt_indices = np.where(distances_to_P_current_global < 0.5)[0] # Points within 0.5m 
        
        if close_gt_indices.shape[0] > 0:
            print(f"\nGT points from 'gt_all_labeled_pts_structured' close to P_current_global (Point 3028):")
            print(all_gt_struct[['x', 'y', 'z', 'velocity_x', 'velocity_y', 'instance_token']][close_gt_indices])
            
            # Calculate their speed
            close_gt_speeds = np.sqrt(all_gt_struct['velocity_x'][close_gt_indices]**2 + all_gt_struct['velocity_y'][close_gt_indices]**2)
            print(f"Speeds of these close GT points: {np.round(close_gt_speeds, 2)}")
        else:
            print(f"\nNo points found in 'gt_all_labeled_pts_structured' within 0.5m of P_current_global.")
    else:
        print("\n'gt_all_labeled_pts_structured' is None or empty.")

if gt_static_points_global is not None and gt_static_points_global.shape[0] > 0 :
    points_to_plot_deep_dive["GT_STATIC_Points_CurrentDI_Global"] = (gt_static_points_global, 0x00AA00, 0.04) # Dark Green
if gt_dynamic_points_global is not None and gt_dynamic_points_global.shape[0] > 0:
    points_to_plot_deep_dive["GT_DYNAMIC_Points_CurrentDI_Global"] = (gt_dynamic_points_global, 0xAA0000, 0.04) # Dark Red

# --- Display Plot and Print Info ---
# from src.visualization.k3d_visualizer import create_k3d_plot_with_points # Ensure this is imported in your notebook
plot_deep_dive = create_k3d_plot_with_points(points_to_plot_deep_dive, 
                                           f"Deep Dive Pt {target_pt_idx_deep_dive} vs HistDI TS {historical_di_ts_deep_dive}")
if plot_deep_dive: 
    plot_deep_dive.display()
else:
    print("Failed to create K3D plot for deep dive.")

print(f"K3D plot for deep dive can be generated.")
print(f"  P_current_global (Magenta, idx {target_pt_idx_deep_dive}): {np.round(P_current_global,3)}")
print(f"  P_projected_global (Red, current point as seen from hist sensor): {np.round(P_projected_global,3)}")
print(f"  P_cand1_hist_global (Yellow, hist point idx {hist_cand_idx1_deep_dive}): {np.round(P_cand1_hist_global,3)}")
print(f"  P_cand2_hist_global (Cyan, hist point idx {hist_cand_idx2_deep_dive}): {np.round(P_cand2_hist_global,3)}")
print(f"  Projected depth of current point from hist sensor (Red point's basis): {depth_proj:.3f}m")

if historical_di_object_deep_dive.local_sph_coords_for_points is not None:
    print(f"  Candidate 1 (Yellow) depth from hist sensor: {historical_di_object_deep_dive.local_sph_coords_for_points[hist_cand_idx1_deep_dive][2]:.3f}m")
    print(f"  Candidate 2 (Cyan) depth from hist sensor: {historical_di_object_deep_dive.local_sph_coords_for_points[hist_cand_idx2_deep_dive][2]:.3f}m")
else:
    print("  Historical DI's local_sph_coords_for_points not available to print candidate depths.")
    
    
    
# Cell 5.3: Inspecting Event Test (Test 1) and Final Labels

import numpy as np # If not already imported in this cell block
from src.core.constants import OcclusionResult # If not already imported

if processed_target_di and \
   processed_target_di.mdet_labels_for_points is not None and \
   processed_target_di.raw_occlusion_results_vs_history is not None and \
   processed_target_di.original_points_global_coords is not None and \
   config: # Added original_points_global_coords check for safety
    
    print("\n--- Inspecting Event Test (Test 1 - Perpendicular) and Final Labels ---")
    
    final_labels_target_di = processed_target_di.mdet_labels_for_points
    history_array_target_di = processed_target_di.raw_occlusion_results_vs_history
    
    event_test_cfg = config.get('event_tests', {})
    test1_M1_threshold = event_test_cfg.get('test1_M1_threshold', 2)
    
    # --- Define points to inspect, ensuring Point 3028 is included ---
    # Use a mix of general points and specific points of interest
    num_pts_total = processed_target_di.original_points_global_coords.shape[0]
    
    # Base sample points
    sample_pt_indices_to_inspect = [0, 100, 1000, 5000, 10000] 
    # Add specific points of interest
    points_of_interest = [1786, 2142, 3028, 3053, 3054] # From MCC log
    
    # Combine and ensure they are valid and unique
    combined_indices = sample_pt_indices_to_inspect + points_of_interest
    valid_indices = sorted(list(set([idx for idx in combined_indices if idx < num_pts_total])))
    
    if not valid_indices:
        print("No valid sample point indices to inspect for final labels and event tests.")
    
    for pt_idx in valid_indices:
        print(f"\nPoint Index {pt_idx}:")
        print(f"  Global Coords: {np.round(processed_target_di.original_points_global_coords[pt_idx],3).tolist()}")

        # --- Ground Truth Check (copied from Cell 5.2 for context here too) ---
        if gt_labels_target_sweep: # Assuming gt_labels_target_sweep is available from Cell 5.2
            point_to_check_global_for_gt = processed_target_di.original_points_global_coords[pt_idx]
            dist_thresh_sq = 0.1**2 
            
            is_gt_dynamic = False
            if gt_labels_target_sweep.get('gt_dynamic_pts') is not None and gt_labels_target_sweep['gt_dynamic_pts'].shape[0] > 0:
                distances_sq_dyn = np.sum((gt_labels_target_sweep['gt_dynamic_pts'] - point_to_check_global_for_gt)**2, axis=1)
                if np.any(distances_sq_dyn < dist_thresh_sq):
                    is_gt_dynamic = True
                    print(f"  GT INFO: Point {pt_idx} is CLOSE to a GT DYNAMIC point.")

            is_gt_static = False
            if not is_gt_dynamic and gt_labels_target_sweep.get('gt_static_pts') is not None and gt_labels_target_sweep['gt_static_pts'].shape[0] > 0:
                distances_sq_stat = np.sum((gt_labels_target_sweep['gt_static_pts'] - point_to_check_global_for_gt)**2, axis=1)
                if np.any(distances_sq_stat < dist_thresh_sq):
                    is_gt_static = True
                    print(f"  GT INFO: Point {pt_idx} is CLOSE to a GT STATIC point.")
            
            if not is_gt_dynamic and not is_gt_static:
                print(f"  GT INFO: Point {pt_idx} is UNLABELED as dynamic/static or not close to any GT point.")
        else:
            print("  GT INFO: Not available for this point/sweep (gt_labels_target_sweep not loaded).")
        # --- End Ground Truth Check ---

        # Event Test (Perpendicular)
        occluding_count = np.sum(history_array_target_di[pt_idx, :] == OcclusionResult.OCCLUDING_IMAGE.value)
        passed_event_test_perp = (occluding_count >= test1_M1_threshold)
        print(f"  Event Test (Perpendicular): Occluding count = {occluding_count} (Threshold M1 = {test1_M1_threshold}) -> Passed: {passed_event_test_perp}")
        
        # Retrieve the final label assigned by process_and_label_di
        final_label_val = final_labels_target_di[pt_idx]
        final_label_enum = OcclusionResult(final_label_val)
        print(f"  Final Assigned Label: {final_label_enum.name} ({final_label_val})")
        
        # Raw history for context
        # history_for_pt_readable = [OcclusionResult(val).name for val in history_array_target_di[pt_idx, :]]
        # print(f"  Raw Occlusion History: {history_for_pt_readable}") # Can be verbose, uncomment if needed

else:
    print("Final label and event test inspection skipped: Target DI not fully processed or missing required data.")
    
    
    
if nusc and TARGET_SWEEP_DATA_DICT and processed_target_di and processed_target_di.mdet_labels_for_points is not None:
    print("\n--- Visualizing Final M-Detector Output vs. Ground Truth (K3D) ---")
    
    # 1. Load GT Labels for the target sweep
    gt_points_data = load_gt_labels_for_sweep_from_hdf5(
        nusc,
        target_scene_rec['name'],
        TARGET_SWEEP_DATA_DICT,
        GT_LABELS_HDF5_DIR,
        config.get('validation',{}).get('vel_threshold', 0.5) # Use vel_thresh from config
    )
    
    # 2. Prepare M-Detector points from the processed_target_di
    mdet_all_pts_global = processed_target_di.original_points_global_coords
    mdet_all_labels = processed_target_di.mdet_labels_for_points
    
    points_for_k3d_comparison = {}
    
    # Add Raw LiDAR points from the current sweep (can be downsampled)
    raw_points_global_current_sweep = processed_target_di.original_points_global_coords
    if raw_points_global_current_sweep is not None and raw_points_global_current_sweep.shape[0] > 0:
         points_for_k3d_comparison["Raw_LiDAR_TargetSweep"] = (raw_points_global_current_sweep[::5], 0xAAAAAA, 0.02) # Grey, small

    # Add GT points
    # if gt_points_data:
    #     if gt_points_data['gt_dynamic_pts'].shape[0] > 0:
    #         points_for_k3d_comparison["GT_Dynamic"] = (gt_points_data['gt_dynamic_pts'], 0x0000FF, 0.07) # Blue
    #     if gt_points_data['gt_static_pts'].shape[0] > 0:
    #         points_for_k3d_comparison["GT_Static"] = (gt_points_data['gt_static_pts'], 0x00FFFF, 0.05) # Cyan
            
    # Add M-Detector classified points
    if mdet_all_pts_global is not None and mdet_all_labels is not None:
        labels_map_k3d = {
            OcclusionResult.OCCLUDING_IMAGE.value: ("MDet_Dynamic", 0x00FF00, 0.07), # Green
            OcclusionResult.OCCLUDED_BY_IMAGE.value: ("MDet_Occluded", 0xFFA500, 0.06), # Orange
            OcclusionResult.EMPTY_IN_IMAGE.value: ("MDet_Empty", 0xFFFF00, 0.04),       # Yellow
            OcclusionResult.UNDETERMINED.value: ("MDet_Undetermined", 0x808080, 0.03) # Darker Grey
        }
        for label_val, (name, color, size) in labels_map_k3d.items():
            mask = (mdet_all_labels == label_val)
            if np.any(mask):
                points_for_k3d_comparison[name] = (mdet_all_pts_global[mask], color, size)
                
    plot_comparison = create_k3d_plot_with_points(
        points_for_k3d_comparison,
        f"GT vs MDet: {target_scene_rec['name']} - Sweep {TARGET_LIDAR_SD_TOKEN[:8]}"
    )
    if plot_comparison:
        display(plot_comparison)
    else:
        print("Failed to generate K3D comparison plot.")
        
else:
    print("Comparison visualization skipped: Missing NuScenes, target sweep, or processed M-Detector data.")

    